# 99_explore — optional read-only exploration support

Use this optional support notebook for discovery, profiling, troubleshooting, investigation, and ad hoc analysis. The required delivery path remains: `01_agreement` → `02_pipeline` → `03_governance`.

`99_explore` can read selected agreement context and existing catalogue context to help you investigate a source table. It does **not** approve agreements, enforce guardrails, write pipeline metadata, register delivery state, promote outputs, or mutate governance metadata.

Keep repeatable transformation logic in `02_pipeline`. Keep approval and review workflows in `01_agreement` or `03_governance`.


## 01 Configure environment


In [ ]:
%run 00_env_config


## 02 Import functions


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    widget_pipeline_bootstrap,
)


## 03 Select agreement

Select an agreement for investigation context only. This optional exploration notebook does not register itself as part of the delivery workflow.


In [ ]:
PIPELINE = widget_pipeline_bootstrap(
    notebook_type="99_explore",
    select_agreement=True,
    register_notebook=False,
    read_only=True,
)

AGREEMENT = PIPELINE.agreement
AGREEMENT


## 04 Read source

Edit the source table values below. Keep optional file and warehouse examples commented unless you need them.


In [ ]:
# ============================================================
# User inputs
# Change this section for your source table
# ============================================================
source_table_name = "your_source_table_name"
source_schema = SOURCE_SCHEMA
source_target = "source"

source_df = read_lakehouse_table(
    source_table_name,
    schema=source_schema,
    target=source_target,
    spark_session=spark,
)

source_df.printSchema()
source_df.limit(20).show(truncate=False)

# Optional examples — uncomment only when needed.
# CSV file:
# source_df = read_lakehouse_csv("Files/input/example.csv", spark_session=spark, header=True)
# Excel file:
# source_df = read_lakehouse_excel("Files/input/example.xlsx", sheet_name=0, spark_session=spark)
# Parquet file:
# source_df = read_lakehouse_parquet("Files/input/example.parquet", spark_session=spark)
# Warehouse table:
# source_df = read_warehouse_query("SELECT * FROM dbo.source_table WHERE business_date >= '2026-01-01'", target="warehouse", spark_session=spark)


## 05 Read metadata catalogue

Read `METADATA_DATA_CATALOGUE` directly from the configured metadata target, then filter in-notebook for the selected source table and optional agreement context. If no catalogue evidence exists yet, the filtered DataFrame is simply empty.


In [ ]:
metadata_catalogue = read_lakehouse_table(
    "METADATA_DATA_CATALOGUE",
    target="metadata",
    schema=METADATA_SCHEMA,
    spark_session=spark,
)

latest_catalogue = metadata_catalogue.filter(F.col("table_name") == source_table_name)

agreement_id = str((AGREEMENT or {}).get("agreement_id") or "").strip()
if agreement_id and "agreement_id" in latest_catalogue.columns:
    latest_catalogue = latest_catalogue.filter(F.col("agreement_id") == agreement_id)

contract_version = str((AGREEMENT or {}).get("agreement_contract_version") or (AGREEMENT or {}).get("contract_version") or "").strip()
if contract_version and "contract_version" in latest_catalogue.columns:
    latest_catalogue = latest_catalogue.filter(F.col("contract_version") == contract_version)

latest_catalogue


## 06 Optional profile, no write

Set `RUN_PROFILE` to `True` to profile the DataFrame in this notebook session. This profile is local exploratory output only; it does not update metadata tables, create catalogue evidence, approve governance, or enforce guardrails.


In [ ]:
RUN_PROFILE = True

if RUN_PROFILE:
    profile_df = profile_dataframe(
        source_df,
        table_name=source_table_name,
    )

    profile_df.show(50, truncate=False)


## 07 Self exploration

Use this scratch section for focused checks while investigating the source table.

Suggested checks:
- null checks
- distinct counts
- duplicate checks
- date range checks
- mapping validation
- sample inspection
- joins to reference data

Keep final, repeatable transformation logic in `02_pipeline`.


In [ ]:
# Add focused exploratory checks here.
# Examples:
# - source_df.groupBy("status").count().show()
# - source_df.select("business_date").summary().show()
# - source_df.groupBy("natural_key").count().filter("count > 1").show()

display(source_df.limit(100))
